# 최적화 분할 실습 (SLIC Superpixel & N-Cut)

이 실습은 **SLIC(Superpixel Linear Iterative Clustering)**을 통해 영상을 잘게 쪼갠 후, **N-Cut(Normalized Cut)** 알고리즘을 적용하여 유사한 영역끼리 최적화하여 병합하는 과정을 다룹니다.

In [ ]:
import cv2
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from skimage.segmentation import slic, mark_boundaries
from skimage.color import label2rgb
from skimage import graph  # N-Cut 계산을 위한 그래프 모듈

# 1. 샘플 이미지 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg'
urllib.request.urlretrieve(url, 'butterfly.jpg')
img = cv2.imread('butterfly.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. SLIC 슈퍼픽셀 분할 (Oversegmentation 상태, 400조각)
labels = slic(img_rgb, n_segments=400, compactness=10, sigma=1, start_label=1)

# 3. 슈퍼픽셀들 간의 인접 관계와 색상 차이를 계산하여 그래프 생성
g = graph.rag_mean_color(img_rgb, labels)

# 4. N-Cut (Normalized Cut) 적용
# thresh: 잘라낼 기준값. 낮을수록 더 많이 쪼개지고, 높을수록 크게 뭉침
# num_cuts: 반복해서 자를 횟수
nc_labels = graph.cut_normalized(labels, g, thresh=1.0, num_cuts=2)

# 5. 결과 시각화 준비
slic_avg = label2rgb(labels, img_rgb, kind='avg')  # SLIC 결과 (평균 색상)
ncut_avg = label2rgb(nc_labels, img_rgb, kind='avg')  # N-Cut 결과 (평균 색상)

# 6. 최종 출력
plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
plt.title("1. Original")
plt.imshow(img_rgb)
plt.axis('off')

plt.subplot(1, 4, 2)
plt.title("2. SLIC Boundaries")
plt.imshow(mark_boundaries(img_rgb, labels))
plt.axis('off')

plt.subplot(1, 4, 3)
plt.title("3. SLIC Avg Color")
plt.imshow(slic_avg.astype('uint8'))
plt.axis('off')

plt.subplot(1, 4, 4)
plt.title("4. N-Cut Result")
plt.imshow(ncut_avg.astype('uint8'))
plt.axis('off')

plt.tight_layout()
plt.show()